# FTRL example

One FTRL batch: data -> fixed-model predictions -> covariate-oblivious selection -> label -> inference.

Run from `examples/simulation/`, so that `dataset` and `model` import directly.

In [1]:
from model.judge import JudgeModel
from openppi import Mean, get_method, select_for_labeling
from dataset.judge_sim import make_judge_data

In [2]:
# 1. Separate features and labels from the data provider's DataFrame (rows in arrival order).
data = make_judge_data(n=2000, seed=0)
features = data.drop(columns='label')
judge_cols = [i for i, name in enumerate(features.columns) if name.startswith('judge')]
features = features.to_numpy()
y = data['label'].to_numpy()

In [3]:
# 2. Select the target and predict with a fixed model (any already trained model works).
target = Mean()
recipe = get_method('FTRL')         # sampling: FTRLDesign; inference: IPW, as for ASI
model = JudgeModel(columns=judge_cols)
predictions = model.predict(features)

In [4]:
# 3. FTRL sets p_t from the oracle Phi of earlier rows only; Phi ~ E[(Y - f)^2 | X],
#    here f(1 - f) for a calibrated binary model. p_t starts at tau/8 and rises to tau.
request = select_for_labeling(predictions, target=target, design=recipe.sampling(),
                              scores=predictions * (1 - predictions), budget=400, random_state=42)
print('Rows to label:', len(request.indices), '| first p_t:', request.propensities[:3].round(3))

Rows to label: 408 | first p_t: [0.025 0.2   0.2  ]


In [5]:
# 4. Human labeling. The simulation looks up y; in practice, export request.indices to annotators.
labels = y[request.indices]

In [6]:
# 5. The estimator and interval are those of ASI (IPW).
record = request.with_labels(labels)
result = recipe.inference(record, target=target, alpha=.05)
print('Estimate:', result.estimate)
print('Confidence interval:', result.confidence_interval)
print('Diagnostics:', result.diagnostics)

Estimate: 0.49991093931431896
Confidence interval: (0.45182466579515185, 0.5479972128334861)
Diagnostics: {'audits': 408, 'sum_sampling_probabilities': 399.82500000000005, 'min_pi': 0.025, 'max_pi': 0.2, 'prediction_weight': 1.0, 'weight_strategy': 'fixed', 'variance_design': 'active_terminal_plugin'}
